In [2]:
import os

In [3]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-'

In [6]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/Muhammedsuraj/End-to-End-ML-Classification-Project-with-MLflow-.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="Muhammedsuraj"
os.environ["MLFLOW_TRACKING_PASSWORD"]="44abd337adfd5d5cedf082ab7d4935e98db9efdf"

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    test_target_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [8]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            test_target_data_path=config.test_target_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/Muhammedsuraj/End-to-End-ML-Classification-Project-with-MLflow-.mlflow",
           
        )

        return model_evaluation_config


In [12]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [21]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        accuracy = accuracy_score(actual, pred)
        f1 = f1_score(actual, pred)
        recall = recall_score(actual, pred)
        precision = precision_score(actual, pred)
        class_report = classification_report(actual, pred)
        return accuracy, f1, recall, precision, classification_report
    


    def log_into_mlflow(self):

        test_x = pd.read_csv(self.config.test_data_path)
        test_y = pd.read_csv(self.config.test_target_data_path).iloc[:,0]
        model = joblib.load(self.config.model_path)


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            pred = model.predict(test_x)

            (accuracy, f1, recall, precision, class_report) = self.eval_metrics(test_y, pred)
            
            # Saving metrics as local
            scores = {"accuracy": accuracy, "f1": f1, "recall": recall, "precision": precision}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("f1", f1)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("precision", precision)
            #mlflow.log_metric("classification_report", class_report)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="RandomForestClassifier")
            else:
                mlflow.sklearn.log_model(model, "model")

    


In [22]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-02-26 04:20:20,696: INFO: common: created directory at: artifacts]
[2026-02-26 04:20:20,700: INFO: common: created directory at: artifacts/model_evaluation]


[2026-02-26 04:20:21,435: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/02/26 04:20:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Projects\End-to-End-ML-Classification-Project-with-MLflow-\venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
Successfully registered model 'RandomForestClassifier'.
2026/02/26 04:21:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForestClassifier, version 1
Created version '1' of model 'RandomForestClassifier'.


🏃 View run languid-goose-45 at: https://dagshub.com/Muhammedsuraj/End-to-End-ML-Classification-Project-with-MLflow-.mlflow/#/experiments/0/runs/0025ba83d53046faa7f2698a8d99da9a
🧪 View experiment at: https://dagshub.com/Muhammedsuraj/End-to-End-ML-Classification-Project-with-MLflow-.mlflow/#/experiments/0
